In [1]:
# INIT: Run once at the beginning
from pprint import pprint
from collections import defaultdict
import numpy as np
import sqlite3
import os
import pandas as pd

class TrainRun:
    def __init__(self, run_id, gpus):
        self.run_id = run_id
        self.is_colocated = "nonswitch" not in run_id
        # If the first type is a float, interpreted as quantile. If int, interpreted as absolute seconds. If str, interpreted as timer name.
        if self.is_colocated:
            self.gpus = {
                # "mixed": [0],
                "mixed": [0,1,2,3],
            }
        else:
            self.gpus = {
                "inference": [0,1],
                "training": [2,3]
            }
        self.sql_t_series = None
        self.nsmi_metrics = None
        self.cpu_metrics = None
        self.timers = {}
        self.run_dir = os.path.join(ROOT_DIR, OUTPUT_DIR, run_id)

OUTPUT_DIR = "output"
ROOT_DIR = "/local_ssd1/mborjigi/skyrl/SkyRL/profile"
RUNS = {}


In [2]:
# SELECT RUNS: Run whenever we change the runs to visualize
ALL_MIXED = {"mixed": [0,1,2,3]}
RUNS_TO_VIZ = [
    # ("swe_7b_4gpu_generate_dp1_10step", ALL_MIXED),
    # ("swe_7b_4gpu_generate_dp1_10step_1", ALL_MIXED),
    # ("swe_14b_4gpu_generate_40step", ALL_MIXED),
    # ("sql_7b_4gpu_32_3gen_again", ALL_MIXED),
    # ("sql_7b_4gpu_64", ALL_MIXED),
    # ("swe_14b_4gpu_generate_40step", ALL_MIXED),
    ("sql_7b_4gpu_debugoom", ALL_MIXED),
    # ("sql_7b_4gpu_128_again", ALL_MIXED),
]
NEW_RUNS = []
for run_id, gpus in RUNS_TO_VIZ:
    if run_id not in RUNS:
        RUNS[run_id] = TrainRun(run_id, gpus)
        NEW_RUNS.append(run_id)
        print(f"Added run {run_id}")
for run_id in list(RUNS.keys()):
    if run_id not in [r for r, _ in RUNS_TO_VIZ]:
        del RUNS[run_id]
        print(f"Removed run {run_id}")

Added run sql_7b_4gpu_debugoom


In [3]:
# HELPERS: Run to adjust implementation of helper functions
import matplotlib.pyplot as plt
import time
from contextlib import contextmanager

@contextmanager
def CtxTimer(name: str):
    # We need to re-import because I have "time" as a global
    # variable lmfaooooo
    import time
    start = time.time()
    try:
        yield None
    finally:
        # Code to release resource, e.g.:
        print(f"'{name}' took {time.time() - start:.2f} seconds")

def plot_histogram(ax, data, title, xlabel, ylabel):
    data = np.array(data)
    print(f"Data stats for '{title}': mean={data.mean():.2f}, median={np.median(data):.2f}, "
          f"std={data.std():.2f}, min={data.min():.2f}, max={data.max():.2f}")
    
    # Use object-oriented API (ax.method) instead of plt.method
    ax.hist(data, bins=20, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True)

def plot_gantt_chart(ax, names, start_times, durations, title, xlabel, ylabel):
      # Plot horizontal bars
      # y=Request ID, width=Duration, left=Start Time
      print(names)
      print(durations)
      print(start_times)
      bars = ax.barh(
            y=names, 
            width=durations, 
            left=start_times, 
            color='#4db6ac', 
            edgecolor='black', 
            alpha=0.8
      )

      # 3. Formatting
      ax.invert_yaxis()  # Put the first request at the top
      ax.set_xlabel(xlabel)
      ax.set_ylabel(ylabel)
      ax.set_title(title)
      ax.grid(axis='x', linestyle='--', alpha=0.5)

      # Optional: Add duration labels next to bars
      for bar in bars:
            width = bar.get_width()
            ax.text(
                  bar.get_x() + width + 0.1, 
                  bar.get_y() + bar.get_height()/2, 
                  f"{width:.2f}s", 
                  va='center', 
                  fontsize=9
            )
from typing import Dict, List, Tuple
def plot_broken_bars(data: Dict[str, List[Tuple[str, str, float, float]]], title: str, xlabel: str, ax):
      x_max = -1
      keys_ordered = sorted(data.keys(), reverse=True) # Reverse to have the first lane at the top
      # TODO fix this shite lmao
      for i, k in enumerate(keys_ordered):
            intervals = data[k]
            colors = [c for _, c, _, _ in intervals]
            times = [(start, duration) for _, start, duration in intervals]
            for start, duration in times:
                  x_max = max(x_max, start + duration)
            ax.broken_barh(times, (i*5+2, 4), facecolors=colors, label=k)
      # # Lane 1: CPU (Y=30, Height=8)
      # ax.broken_barh(cpu_usage, (30, 8), facecolors='tab:blue', label='CPU')

      # # Lane 2: GPU (Y=20, Height=8)
      # ax.broken_barh(gpu_usage, (20, 8), facecolors='tab:orange', label='GPU')

      # # Lane 3: Disk (Y=10, Height=8)
      # ax.broken_barh(disk_io,   (10, 8), facecolors='tab:green', label='Disk')

      # 4. Formatting
      ax.set_ylim(0, len(data)*5+5)
      ax.set_xlim(0, x_max)
      ax.set_xlabel(xlabel)
      yticks = []
      yticklabels = []
      for i, lane_name in enumerate(keys_ordered):
            yticks.append(i*5 + 4)
            yticklabels.append(lane_name)
      ax.set_yticks(yticks)
      ax.set_yticklabels(yticklabels)
      ax.grid(True, axis='x', linestyle='--', alpha=0.5)
      ax.set_title(title)
      task_and_color_set = set()
      for intervals in data.values():
        for task_name, color, _, _ in intervals:
            task_and_color_set.add((task_name, color))
      handles = [plt.Line2D([0], [0], color=color, lw=4) for _, color in task_and_color_set]
      labels = list(task_name for task_name, _ in task_and_color_set)
      ax.legend(handles, labels)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def request_id_sort_key(req_id: str):
    # Example req_id: "0_1_1e8ba6d09fc14028807b434925529db7" or just "0_1"
    parts = req_id.split("_")
    if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
        return (int(parts[0]), int(parts[1]))
    else:
        return req_id  # Fallback to string sorting if format is unexpected

def plot_broken_bars_plotly(data: Dict[str, List[Tuple[str, str, float, float]]], title: str):
    """
    data: Dict[agent_name, List[(task_name, color, start_time_sec, duration_sec)]]
    """
    # Create subplots: Top for concurrency, Bottom for Gantt
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.02, # Reduced gap
        row_heights=[0.1, 0.9], # Shrink top plot (concurrency)
        specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
    )
    
    legend_shown = set()
    agents = sorted(data.keys(), key=request_id_sort_key, reverse=True)

    # 1. Plot Broken Bars (Gantt)
    for agent_name in agents:
        intervals = data[agent_name]
        for task_name, color, start, duration in intervals:
            name = str(task_name) 
            showlegend = name not in legend_shown
            
            fig.add_trace(
                go.Bar(
                    x=[duration],
                    y=[agent_name],
                    width=0.4,
                    base=[start],
                    orientation="h",
                    marker_color=color,
                    name=task_name,
                    showlegend=showlegend,
                    hovertemplate=(
                        "Trajectory=%{y}<br>"
                        "Task=%{fullData.name}<br>"
                        "Start=%{base:.2f}s<br>"
                        "Duration=%{x:.2f}s<extra></extra>"
                    ),
                    legendgroup=task_name # Group legend items
                ),
                row=2, col=1
            )
            legend_shown.add(task_name)
    
    # 2. Plot Concurrency (Number of active intervals)
    events = []
    for agent_name, intervals in data.items():
        for task_name, color, start, duration in intervals:
             # Add start event (+1) and end event (-1)
             events.append((start, 1))
             events.append((start + duration, -1))
    
    events.sort(key=lambda x: x[0])
    
    if events:
        x_concurrency = [events[0][0]]
        y_concurrency = [0]
        current_active = 0
        
        for t, change in events:
            # Add point before change to create step effect
            x_concurrency.append(t)
            y_concurrency.append(current_active)
            
            current_active += change
            
            # Add point after change
            x_concurrency.append(t)
            y_concurrency.append(current_active)
            
        fig.add_trace(
            go.Scatter(
                x=x_concurrency,
                y=y_concurrency,
                mode='lines',
                name='Active Тrajectories',
                line=dict(color='black', width=1),
                fill='tozeroy',
                fillcolor='rgba(0,0,0,0.1)',
                showlegend=False,
                log_y=True,
            ),
            row=1, col=1
        )

    fig.update_layout(
        title=title,
        yaxis2_title="Trajectory",
        yaxis_title="Active Тrajectories",
        barmode="overlay", 
        bargap=0.2,
        height=max(300, 20 * len(agents) + 200), # Reduced height calculation
        width=1500,
        legend=dict(
            yanchor="top",
            y=0.87,  # Place it near the top of the bottom plot (since row 1 height is small)
            xanchor="right",
            x=0.99,
            bgcolor="rgba(255, 255, 255, 0.5)" # Semi-transparent background
        )
    )
    
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.1)", row=2, col=1)
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.1)", row=1, col=1)
    fig.update_yaxes(categoryorder="array", categoryarray=agents, row=2, col=1)
    # fig.update_yaxes(type="log", row=1, col=1)
    
    return fig

import typing as T
from typing import Optional
class PlotData(T.NamedTuple):
    metric_name: str
    color: str
    line_style: str
    data: pd.Series

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import Dict, List, Tuple

def plot_compact_gpu_rollouts(data: Dict[str, List[Tuple[str, str, float, float]]], title: str, extra_plots: Dict[str, List[PlotData]] = {}, vlines: List[Tuple[str, float, Optional[float]]] = []):
    """
    data: Dict[gpu_rank, List[(task_name, color, start_time_sec, duration_sec)]]

    Features:
    - Rollout affinity: each unique task_name is globally assigned a fixed lane index
      so the same rollout never migrates between lanes across GPUs.
    - Per-GPU rows: only lanes *actually used* by each GPU are emitted as rows,
      so there are no empty rows.
    - Single GPU label per section (no per-lane tick labels).
    - Dense layout: bars nearly touch (bargap=0, width≈1).
    """
    # Hardcoding "gpu 999" to make only 1 lane
    data[""] = []
    for k in list(data.keys()):
        if k != "":
            data[""].extend(data[k])
            del data[k]
    gpus = sorted(data.keys(), key=lambda x: int(x) if x.isdigit() else x)

    # ── 1. Global affinity: assign each req_id a fixed lane ──────────────────
    # Use the merged time span across all GPUs so the packing is stable.
    req_span: Dict[str, tuple] = {}
    for intervals in data.values():
        for task_name, _, start, duration in intervals:
            end = start + duration
            if task_name not in req_span:
                req_span[task_name] = (start, end)
            else:
                s, e = req_span[task_name]
                req_span[task_name] = (min(s, start), max(e, end))

    sorted_reqs = sorted(req_span.items(), key=lambda x: request_id_sort_key(x[0]))
    req_to_lane: Dict[str, int] = {}
    lane_ends: List[float] = []
    for task_name, (start, end) in sorted_reqs:
        placed = False
        for lane_idx, lane_end in enumerate(lane_ends):
            if lane_end <= start:
                lane_ends[lane_idx] = end
                req_to_lane[task_name] = lane_idx
                placed = True
                break
        if not placed:
            req_to_lane[task_name] = len(lane_ends)
            lane_ends.append(end)

    # ── 2. Build ordered y-label list (only occupied lanes per GPU) ───────────
    # For each GPU we emit one row per lane that has ≥1 request on that GPU.
    # Lanes are ordered by their global lane index so affinity is visually
    # consistent (lane 0 is always the topmost row in each GPU block).
    SPACER_LANES = 1
    y_labels_ordered: List[str] = []
    gpu_tick_labels: List[str] = []
    gpu_tick_vals:  List[str] = []

    for i, gpu in enumerate(gpus):
        used_lanes = sorted(set(req_to_lane[task_name] for task_name, _, _, _ in data[gpu]))
        lane_labels = [f"G{gpu}_L{lane}" for lane in used_lanes]
        if not lane_labels:
            continue
        y_labels_ordered.extend(lane_labels)
        mid = lane_labels[len(lane_labels) // 2]
        gpu_tick_vals.append(mid)
        if gpu == "":
            gpu_tick_labels.append("")
        else:
            gpu_tick_labels.append(f"GPU {gpu}")
        if i < len(gpus) - 1:
            for j in range(SPACER_LANES):
                y_labels_ordered.append(f"__sp_{i}_{j}__")

    # Reverse so GPU 0 is at the top.
    y_labels_ordered.reverse()

    # ── 3. Figure setup ───────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2 + len(extra_plots), cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        row_heights=[0.33]*3,
        # row_heights=([0.1, 0.7] + [0.2 / len(extra_plots)] * len(extra_plots)) if extra_plots else [0.1, 0.9],
        specs=[[{"secondary_y": False}]] * (2 + len(extra_plots)),
    )
    
    for i, (extra_plot_name, plot_data_list) in enumerate(extra_plots.items()):
        for plot_data in plot_data_list:
            fig.add_trace(
                go.Scatter(
                    x=plot_data.data.index,
                    y=plot_data.data.values,
                    mode="lines",
                    name=plot_data.metric_name,
                    line=dict(color=plot_data.color, width=2, dash=plot_data.line_style),
                    showlegend=False
                ),
                row=3 + i, col=1,
            )

    legend_shown: set = set()

    # ── 4. Plot bars ──────────────────────────────────────────────────────────
    for gpu in gpus:
        for task_name, color, start, duration in data[gpu]:
            lane_idx = req_to_lane[task_name]
            y_label = f"G{gpu}_L{lane_idx}"
            showlegend = task_name not in legend_shown
            fig.add_trace(
                go.Bar(
                    x=[duration],
                    y=[y_label],
                    base=[start],
                    orientation="h",
                    marker_color=color,
                    marker_line_width=0,
                    name=task_name,
                    showlegend=False,
                    width=0.97,
                    hovertemplate=(
                        f"GPU={gpu}<br>"
                        f"Lane={lane_idx}<br>"
                        "Req=%{fullData.name}<br>"
                        "Start=%{base:.2f}s<br>"
                        "Duration=%{x:.2f}s<extra></extra>"
                    ),
                    legendgroup=task_name,
                ),
                row=2, col=1,
            )
            legend_shown.add(task_name)

    # ── 5. Separator lines between GPU blocks ─────────────────────────────────
    for i in range(len(gpus) - 1):
        spacer_label = f"__sp_{i}_0__"
        fig.add_hline(
            y=spacer_label,
            line_dash="solid",
            line_color="rgba(0,0,0,0.4)",
            line_width=1,
            row=2, col=1,
        )

    # ── 6. Concurrency trace ──────────────────────────────────────────────────
    events = []
    for intervals in data.values():
        for _, _, start, duration in intervals:
            events.append((start, 1))
            events.append((start + duration, -1))
    events.sort(key=lambda x: x[0])

    if events:
        x_conc = [events[0][0]]
        y_conc = [0]
        cur = 0
        for t, change in events:
            x_conc.append(t)
            y_conc.append(cur)
            cur += change
            x_conc.append(t)
            y_conc.append(cur)
        fig.add_trace(
            go.Scatter(
                x=x_conc, y=y_conc,
                mode="lines",
                name="Active Requests",
                line=dict(color="black", width=1),
                fill="tozeroy",
                fillcolor="rgba(0,0,0,0.1)",
                showlegend=False,
            ),
            row=1, col=1,
        )

    for i, extra_plot_name in enumerate(extra_plots.keys()):
        fig.update_yaxes(title_text=extra_plot_name, row=3+i, col=1)
    
    # for i, plot_name in enumerate(extra_plots):
    #     yaxis_titles[f"yaxis{i+3}_titles"] = plot_name

    # ── 6b. Add vlines (vertical lines or shaded areas) through all subplots ───
    color_it = iter(["red", "blue", "green", "purple", "orange"])
    color_map = {}
    for vline in vlines:
        if vline[0] not in color_map:
            color_map[vline[0]] = next(color_it)
        color = color_map[vline[0]]
        if len(vline) == 2:
            # Vertical line: (name, x_value)
            name, x_val = vline
            print(f"Adding vline at x={x_val} with name='{name}'")
            fig.add_vline(
                x=x_val,
                line_dash="dash",
                line_color=color,
                line_width=1,
                annotation_text=name,
                annotation_position="top"
            )
        elif len(vline) == 3:
            # Shaded area: (name, x_start, x_end)
            name, x_start, x_end = vline
            print(f"Adding vrect from x={x_start} to x={x_end} with name='{name}'")
            fig.add_vrect(
                x0=x_start,
                x1=x_end,
                fillcolor=color,
                opacity=0.1,
                layer="below",
                line_width=0,
                annotation_text=name,
                annotation_position="top"
            )

    # ── 7. Layout ─────────────────────────────────────────────────────────────
    PX_PER_LANE = 5
    n_rows = len(y_labels_ordered)
    fig.update_layout(
        title=title,
        yaxis_title="Active Requests",
        barmode="overlay",
        bargap=0,
        height=max(300, PX_PER_LANE * n_rows + 180),
        width=1500,
        plot_bgcolor="white",
        # showlegend=False,
        # legend=dict(
        #     yanchor="top", y=0.87,
        #     xanchor="right", x=0.99,
        #     bgcolor="rgba(255,255,255,0.8)",
        #     bordercolor="rgba(0,0,0,0.2)",
        #     borderwidth=1,
        # ),
    )

    fig.update_yaxes(
        showline=True,
        categoryorder="array",
        categoryarray=y_labels_ordered,
        tickmode="array",
        tickvals=gpu_tick_vals,
        ticktext=gpu_tick_labels,
        row=2, col=1,
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.1)", row=2, col=1)
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.1)", row=1, col=1)
    
    for row in range(1, 3 + len(extra_plots)):
        fig.update_xaxes(
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=True,  # Shows lines on all sides
            row=row, col=1
        )
        fig.update_yaxes(
            showline=True,
            linewidth=1,
            linecolor='black',
            mirror=True,
            row=row, col=1
        )

    fig.update_xaxes(title_text="Time (s)", row=2 + len(extra_plots), col=1)
    return fig

# import random

# def generate_dummy_data():
#     """
#     Generates dummy data for 3 GPUs with varying levels of concurrency.
#     Format: {gpu_rank: [(task_name, color, start, duration), ...]}
#     """
#     data = {}
#     colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
    
#     # GPU 0: Highly concurrent (should create multiple lanes)
#     gpu0_tasks = []
#     for i in range(10):
#         start = random.uniform(0, 10)
#         duration = random.uniform(5, 15)
#         task_id = f"T0_{i}"
#         gpu0_tasks.append((task_id, colors[i % len(colors)], start, duration))
#     data["0"] = gpu0_tasks

#     # GPU 1: Sequential requests (should stay mostly in 1 or 2 lanes)
#     gpu1_tasks = []
#     current_time = 0
#     for i in range(5):
#         duration = random.uniform(8, 12)
#         task_id = f"T1_{i}"
#         # Small 1-second gaps
#         gpu1_tasks.append((task_id, colors[i % len(colors)], current_time, duration))
#         current_time += duration + 1
#     data["1"] = gpu1_tasks

#     # GPU 2: Burst patterns
#     gpu2_tasks = [
#         ("Burst_A1", colors[0], 2, 10),
#         ("Burst_A2", colors[1], 2.5, 9),
#         ("Burst_A3", colors[2], 3, 8),
#         ("Burst_B1", colors[3], 20, 5),
#         ("Burst_B2", colors[4], 21, 5),
#     ]
#     data["2"] = gpu2_tasks

#     return data

# # --- Execution ---
# test_data = generate_dummy_data()
# fig = plot_compact_gpu_rollouts(test_data, "LLM Rollout Performance Test")
# fig.show()


In [4]:
# READ DATA: Run once per run
import re
for runid, run in RUNS.items():
    METRIC_NAMES = [
        "Tensor Active [Throughput %]",
        "GR Active [Throughput %]",
        "SMs Active [Throughput %]",
        "SM Issue [Throughput %]",
        "Compute Warps in Flight [Throughput %]",
        "DRAM Read Bandwidth [Throughput %]",
        "DRAM Write Bandwidth [Throughput %]",
        "PCIe RX Throughput [Throughput %]",
        "PCIe TX Throughput [Throughput %]",
    ]
    # corresponds to CUDA_VISIBLE_DEVICES=1,2,3
    # These are stable somehow idkkkkkkk
    GPU_IDS = {'281479271677952': 1, '281479271677953': 0, '281479271677954': 3, '281479271677955': 2}

    con = sqlite3.connect(os.path.join(run.run_dir, "nsys_report.sqlite"))
    cur = con.cursor()
    eligible_metric_names = ",".join(map(lambda x: f"'{x}'", METRIC_NAMES))
    with CtxTimer(f"{runid} nsys load"):
        metric_id_and_name = dict(cur.execute(
            "SELECT metricId, metricName FROM target_info_gpu_metrics"
        ).fetchall())
        metric_id_name_map = {}
        for mid, mname in metric_id_and_name.items():
            if mname not in METRIC_NAMES:
                continue
            if mid not in metric_id_name_map:
                metric_id_name_map[mid] = mname
            elif metric_id_name_map[mid] != mname:
                print(f"Warning: metricId {mid} maps to multiple names: '{metric_id_name_map[mid]}' and '{mname}'")
        wanted_metric_ids = ",".join(str(mid) for mid in metric_id_name_map.keys())
        wanted_type_ids = ",".join(str(tid) for tid in GPU_IDS.keys())

        # This is actually slower than NOT INDEXED atm
        # The query is the full table. But an index might be useful
        # later and this only takes ~3s longer.
        run.full_df = pd.read_sql_query(f"SELECT timestamp, value, metricId, typeId FROM gpu_metrics WHERE metricId in ({wanted_metric_ids}) AND typeId in ({wanted_type_ids}) AND timestamp > 0", con)
    run.raw_data = defaultdict(dict)
    run.data = defaultdict(dict)
    run.auc = defaultdict(dict)
    run.full_df["timestamp"] = pd.to_datetime(run.full_df["timestamp"], unit='ns')
    run.full_df.sort_values("timestamp", inplace=True)
    grouped = run.full_df.groupby(['typeId', 'metricId'])
    with CtxTimer(f"{runid} data extraction"):
        for (gpu_id, mid), df in grouped:
            mname = metric_id_name_map[mid]
            if str(gpu_id) not in GPU_IDS or mname not in METRIC_NAMES:
                print(f"Skipping GPU ID {gpu_id} Metric {mname} as it's not in the expected list")
                continue
            # df is already a pre-calculated slice; no need for manual filtering
            # Since SQL 'order by timestamp' was used, it's already sorted
            temp_df = df.set_index("timestamp")
            
            run.raw_data[GPU_IDS[str(gpu_id)]][mname] = temp_df["value"]
            # run.auc[real_id][mname] = np.trapz(temp_df["value"], x=temp_df.index.view(np.int64))

    def get_val(d: dict):
        return next(iter(d.values()))
    run.sql_t_series = get_val(get_val(run.raw_data)).index.to_series()
    
    TIME_PATTERN = r"(?:\d{4}-)?\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:\.\d+)?"
    TIME_COST_PATTERN = r"Finished.*time cost: (\d+\.\d+)s"
    PID_PATTERN = r"pid=(\d+)"

    TIME_PATTERN = re.compile(TIME_PATTERN)
    TIME_COST_PATTERN = re.compile(TIME_COST_PATTERN)
    PID_PATTERN = re.compile(PID_PATTERN)
    TIMER_PATTERNS = [
        (name, re.compile(pattern)) for name, pattern in
        [
            ("generate", r"Finished: 'generate'"),
            ("sync_weights", r"Finished: 'sync_weights'"),
            # ("offload_policy_model_to_cpu", r"Finished: 'offload_policy_model_to_cpu'"),
            ("fwd_logprobs_values_reward", r"Finished: 'fwd_logprobs_values_reward'"),
            ("policy_train", r"Finished: 'policy_train'"),
            # ("train_fwd_bwd", r"Finished: 'forward_backward_micro__\d+'"),
            ("train_fwd", r"Finished: 'PolicyWorkerBase._forward_backward_micro__forward'"),
            ("train_bwd", r"Finished: 'FSDPStrategy.backward'"),
            # ("optim_scale_grad", r"Finished: 'optim_step__scale_gradients'"),
        ]
    ]
    # Finished agent loop bubble-roquefort (aka TrajectoryID(instance_id='232', repetition_id=4)) in 44.08 seconds with stop reason stop

    run.raw_timers = {name: [] for name, _ in TIMER_PATTERNS}
    run.timer_pids = {name: [] for name, _ in TIMER_PATTERNS}
    # Timer -> gpu id -> metric name -> auc
    timer_aucs = {name: defaultdict(dict) for name, _ in TIMER_PATTERNS}
    run.first_timer = None
    run.last_timer = None
    target_pids = {timer: set() for timer, _ in TIMER_PATTERNS}
    pidset = set()
    # logger.info(f"Agent loop {readable_trajectory_id} took {time.time() - start:.2f}s, steps: {','.join([f'{t:.2f}' for t in step_times])}, stop reason {stop_reason}")
    trajectory_finished_pattern = re.compile(r"Agent loop (.*) took (\d+\.\d+)s, steps: (\d+\.\d+(?:,\d+\.\d+)*)")
    trajectory_timers = []
    vllm_finished_pattern = re.compile(r"AsyncLLM/(.*):(.*) took (\d+\.\d+)")
    vllm_finished_timers = []
    vllm_state_transition_pattern = re.compile(r"state_transition.*(\d+):(.*):(\w+)->(\w+)")
    # vllm_state_transition_pattern = re.compile(r"state_transition.*(\d+):(\d+_\d+)(?:.*):(\w+)->(\w+)")
    # [36m(AsyncVLLMInferenceEngine pid=2862011)[0m [0;36m(EngineCore_DP0 pid=2862556)[0;0m 2026-03-05 01:45:15.161 | INFO     | vllm.v1.request:state_transition:144 - 0:0_1_1e8ba6d09fc14028807b434925529db7:WAITING->RUNNING
    vllm_state_transitions = []
    swe_agent_pattern = re.compile(r"Agent/(.*):(.*) took (\d+\.\d+)")
    swe_finished_timers = defaultdict(list)
    train_finished_pattern = re.compile(r"Finished: '(.*)_traj_(\d+_\d+)'.*(\d+\.\d+)s")
    train_forward_finished_timers = []
    train_forward_backward_finished_timers = []
    # 'forward_traj_25_1', time cost: 2.74s[0m
    # 'forward_backward_traj_25_1', time cost: 2.74s[0m\
    with open(os.path.join(run.run_dir, "output.log"), errors="ignore") as f, CtxTimer(f"{runid} log parsing"):
        for line in f.readlines():
            def mark_timer(target: str, msg: str):
                if target in line:
                    print(f"[{msg}] detected {target} in line: {line.strip()}")
            mark_timer("AsyncLLM", "start")
            date = re.search(TIME_PATTERN, line)
            if date is None:
                mark_timer("AsyncLLM", "date")
                continue
            date = date.group(0)
            if date.count('-') == 1:
                date = f"2026-{date}"
            time = pd.to_datetime(date)
            swe_match = swe_agent_pattern.search(line)
            if swe_match is not None:
                task_name, agent_name, took = swe_match.groups()
                end = time
                start = end - pd.to_timedelta(float(took), unit='s')
                swe_finished_timers[agent_name].append((task_name, start, end))
            trajectory_match = trajectory_finished_pattern.search(line)
            if trajectory_match is not None:
                traj_id, time_cost, step_times = trajectory_match.groups()
                time_cost = float(time_cost)
                step_times = list(map(float, step_times.split(",")))
                if time_cost * 0.9 > sum(step_times):
                    print(f"WARNING: More than 10% of trajectory {traj_id} is unaccounted for by step times: {time_cost=}, {sum(step_times)=}")
                end_time = time
                start_time = end_time - pd.to_timedelta(time_cost, unit='s')
                trajectory_timers.append((traj_id, start_time, end_time))
            vllm_match = vllm_finished_pattern.search(line)
            if vllm_match is not None:
                mark_timer("AsyncLLM", "vllm_match")
                gpu_id, req_id, timestamp = vllm_match.groups()
                end_time = time
                start_time = end_time - pd.to_timedelta(float(timestamp), unit='s')
                vllm_finished_timers.append((gpu_id, req_id, start_time, end_time))
            else:
                mark_timer("AsyncLLM", "no vllm_match")
            vllm_state_transitions_match = vllm_state_transition_pattern.search(line)
            if vllm_state_transitions_match is not None:
                gpu_id, req_id, from_state, to_state = vllm_state_transitions_match.groups()
                transition_time = time
                vllm_state_transitions.append((gpu_id, req_id, from_state, to_state, transition_time))
            train_finished_match = train_finished_pattern.search(line)
            if train_finished_match is not None:
                timer_name, traj_id, time_cost = train_finished_match.groups()
                end_time = time
                start_time = end_time - pd.to_timedelta(float(time_cost), unit='s')
                if timer_name == "forward":
                    train_forward_finished_timers.append((traj_id, start_time, end_time))
                elif timer_name == "forward_backward":
                    train_forward_backward_finished_timers.append((traj_id, start_time, end_time))
                else:
                    print("ERROR: Unrecognized timer name in train_finished_pattern:", timer_name)
            for timer, pattern in TIMER_PATTERNS:
                if not pattern.search(line):
                    continue
                run.first_timer = run.first_timer if run.first_timer is not None else time
                run.first_timer = min(run.first_timer, time)
                run.last_timer = run.last_timer if run.last_timer is not None else time
                run.last_timer = max(run.last_timer, time)
                # Synthesize a start time by subtracting time cost from end time
                # This is because sometimes the start time is missing or misordered
                time_cost = TIME_COST_PATTERN.search(line)
                if time_cost is not None:
                    pid = PID_PATTERN.search(line)
                    if pid is not None:
                        pid = int(pid.group(1))
                        target_pids[timer].add(pid)
                    end_time = time
                    len_secs = float(time_cost.group(1))
                    start_time = end_time - pd.to_timedelta(len_secs, unit='s')
                    run.raw_timers[timer].append([start_time, end_time])
                    run.timer_pids[timer].append(pid)
                    run.first_timer = min(run.first_timer, start_time)
    run.swe_finished_timers = swe_finished_timers
    run.vllm_finished_timers = vllm_finished_timers
    run.vllm_state_transitions = sorted(vllm_state_transitions, key=lambda x: x[4]) # sort by transition time
    run.trajectory_timers = trajectory_timers
    run.target_pids = target_pids
    run.train_forward_finished_timers = train_forward_finished_timers
    run.train_forward_backward_finished_timers = train_forward_backward_finished_timers

'sql_7b_4gpu_debugoom nsys load' took 1.35 seconds
'sql_7b_4gpu_debugoom data extraction' took 0.04 seconds
[start] detected AsyncLLM in line: (VLLMServerActor pid=3900701) INFO 04-29 02:27:37 [async_llm.py:592] AsyncLLM/None:generate-tokens-acb85a4191637d2e took 4.15
[vllm_match] detected AsyncLLM in line: (VLLMServerActor pid=3900701) INFO 04-29 02:27:37 [async_llm.py:592] AsyncLLM/None:generate-tokens-acb85a4191637d2e took 4.15
[start] detected AsyncLLM in line: (VLLMServerActor pid=3900705) INFO 04-29 02:27:38 [async_llm.py:592] AsyncLLM/None:generate-tokens-ab290f0c200ea00a took 4.31
[vllm_match] detected AsyncLLM in line: (VLLMServerActor pid=3900705) INFO 04-29 02:27:38 [async_llm.py:592] AsyncLLM/None:generate-tokens-ab290f0c200ea00a took 4.31
[start] detected AsyncLLM in line: (VLLMServerActor pid=3900690) INFO 04-29 02:27:38 [async_llm.py:592] AsyncLLM/None:generate-tokens-a56292a2d53eaf5d took 4.58
[vllm_match] detected AsyncLLM in line: (VLLMServerActor pid=3900690) INFO 04

In [5]:
for runid, run in RUNS.items():
    initialized = set()
    finished = set()
    traj_map = defaultdict(list)
    for gpu_id, req_id, start, end, time in run.vllm_state_transitions:
        if start == "INITIAL":
            if req_id in initialized:
                print(f"WARNING: req_id {req_id} has multiple INITIAL states")
            initialized.add(req_id)
        if "FINISHED" in end:
            if req_id in finished:
                print(f"WARNING: req_id {req_id} has multiple FINISHED states")
            finished.add(req_id)
        traj_map[req_id].append((start, end))
    unfinished = initialized - finished
    print(f"Run {runid}: {len(initialized)} initialized, {len(finished)} finished, {len(unfinished)} unfinished")
    for req_id in unfinished:
        print(f"  Unfinished req_id {req_id} with transitions: {traj_map[req_id]}")

Run sql_7b_4gpu_debugoom: 0 initialized, 0 finished, 0 unfinished


In [6]:
# READ CSV DATA: Run when coarse data (cpu and gpu activity) is desired.
for runid, run in RUNS.items():
    file_path = os.path.join(run.run_dir, "gpu_metrics.csv")
    with open(file_path, 'r') as f:
        first_line = f.readline()
        # Clean up the '#' and split by whitespace (or ',' for true CSV)
        columns = first_line.replace('#', '').split()
        # units = f.readline().replace('#', '').split()
        # columns = [i.strip() + " " + j.strip() for i, j in zip(columns, units)]
    nsmi_metrics = pd.read_csv(file_path, sep=r'\s+', comment="#", names=columns)
    nsmi_metrics['timestamp'] = nsmi_metrics['Date'].astype(str) + ' ' + nsmi_metrics['Time']
    nsmi_metrics['timestamp'] = pd.to_datetime(nsmi_metrics['timestamp'], format='%Y%m%d %H:%M:%S')
    nsmi_metrics.set_index('timestamp', inplace=True)
    nsmi_metrics.rename(columns={
        "sm": "compute",
        "mem": "mem_bandwidth",
        "fb": "mem_alloc",
        "rxpci": "pcie_rx",
        "txpci": "pcie_tx",
    }, inplace=True)
    print(nsmi_metrics.columns)
    # plot mem over time
    # nsmi_metrics.plot(x="timestamp", y="mem")
    # nsmi_metrics.plot(x="timestamp", y="sm")
    # nsmi_metrics.plot(x="timestamp", y="gpu")
    # nsmi_metrics.plot(x="timestamp", y="bar1")
    run.raw_nsmi_metrics = nsmi_metrics
    
    file_path = os.path.join(run.run_dir, "cpu_metrics.csv")
    cpu_metrics = pd.read_csv(file_path, parse_dates=["Timestamp"])
    uninteresting_commands = ["ps", "ray::IDLE", "uv"]
    cpu_metrics = cpu_metrics[~cpu_metrics["Command"].isin(uninteresting_commands)]
    cpu_metrics = cpu_metrics[cpu_metrics["Command"] == "ray::FSDPPolicy"]
    cpu_metrics.rename(columns={"Timestamp": "timestamp"}, inplace=True)
    cpu_metrics.set_index('timestamp', inplace=True)
    # print(cpu_metrics["CPU%"].unique())
    # cpu_by_comm = cpu_metrics.groupby("Command").sum()["CPU%"]
    # cpu_by_comm.plot.bar()
    
    # cpu_by_time = cpu_metrics.groupby("timestamp").sum()["CPU%"]
    # cpu_by_time.plot()
    run.raw_cpu_metrics = cpu_metrics
    

Index(['Date', 'Time', 'gpu', 'compute', 'mem_bandwidth', 'enc', 'dec', 'jpg',
       'ofa', 'mem_alloc', 'bar1', 'ccpm', 'pcie_rx', 'pcie_tx'],
      dtype='str')


In [7]:
# FILTER DATASET: create timers and data from raw_timers and raw_data
import os
import re

for runid, run in RUNS.items():
    print(f"Processing run {runid}")
    print(run.swe_finished_timers)
    
    # From the beginning of a train step to the end of the next policy step
    # RUNS[runid].time_frame = (131, 135)
    RUNS[runid].time_frame = (("train_fwd", 0), ("train_bwd", 2))
    RUNS[runid].time_frame = (0.0, 1.0)
    RUNS[runid].time_frame = (("generate", 1), ("sync_weights", 0))
    RUNS[runid].time_frame = (0, 1080)
    RUNS[runid].run_kind = runid

    # RUNS[SINGLE_RUN].time_frame = (0, 100)
    # RUNS[SINGLE_RUN].time_frame = (0.0, 1.0)
    # RUNS[SINGLE_RUN].time_frame = (("train_bwd", 0), ("train_bwd", 0))
    # RUNS[SINGLE_RUN].run_kind = "One GPU, optimizer CPU offload"
    # RUNS["measure-3b"].time_frame = (0.475, 0.48)
    # RUNS["measure-3b"].time_frame = (("train_bwd", 1), ("train_bwd", 1))
    # RUNS["measure-3b"].run_kind = "One GPU, CPU offload"
    # RUNS["fsdpcpu_switch_2gpu_sql_7b_tp1"].time_frame = (("generate", 0), ("policy_train", 2))
    # RUNS["fsdpcpu_switch_2gpu_sql_7b_tp1"].run_kind = "Disaggregation"
    # # From the beginning of a train step to the end of the same train step
    # RUNS["async_nonswitch_4gpu_sql_3b_tp1"].time_frame = (("generate", 2), ("generate", 2))
    # RUNS["async_nonswitch_4gpu_sql_3b_tp1"].run_kind = "Async Disaggregation"
    # # From the beginning of a train step to the end of the next policy step
    # RUNS["switch_4gpu_sql_3b_tp1"].time_frame = (("generate", 1), ("policy_train", 1))
    # RUNS["switch_4gpu_sql_3b_tp1"].run_kind = "Colocation"
    # print(f"{target_pids=}")
    
    base_time = run.first_timer = run.sql_t_series.min()
    last_time = run.last_timer = run.sql_t_series.max()

    print(f"First timer at {run.first_timer}, last timer at {run.last_timer}")
    print(f"Base time at {base_time}")
    if isinstance(run.time_frame[0], int):
        START = base_time + pd.to_timedelta(run.time_frame[0], unit='s')
        END = base_time + pd.to_timedelta(run.time_frame[1], unit='s')
    elif isinstance(run.time_frame[0], float):
        if run.sql_t_series is None:
            diff = last_time - base_time
            START = base_time + pd.to_timedelta(diff.total_seconds() * run.time_frame[0], unit='s')
            END = base_time + pd.to_timedelta(diff.total_seconds() * run.time_frame[1], unit='s')
        else:
            START = run.sql_t_series.quantile(run.time_frame[0])
            END = run.sql_t_series.quantile(run.time_frame[1])
    elif isinstance(run.time_frame[0], tuple):
        assert isinstance(run.time_frame[0][0], str) and isinstance(run.time_frame[0][1], int)
        timer_name, instance_idx = run.time_frame[0]
        START = run.raw_timers[timer_name][instance_idx][0]
        timer_name, instance_idx = run.time_frame[1]
        END = run.raw_timers[timer_name][instance_idx][1]
    else:
        START = base_time
        END = last_time
    assert END > START, f"END time {END} must be after START time {START}"
    print(f"Visualizing data in this range: [{START}, {END}]")
    print(f"As a seconds offset: [{(START - base_time).total_seconds()}, {(END - base_time).total_seconds()}]")

    print(f"Whole application took {last_time - base_time} seconds")
    print("\n\n")
    
    filtered_timers = {}

    target_pid_map = {}
    # for timer in run.target_pids:
    #     print(run.target_pids[timer])
    #     if len(run.target_pids[timer]) == 0:
    #         continue
    #     # Choose max arbitrarily - we just need a consistent pid "hash" across raw_timers that
    #     # may appear in the same process
    #     # Note that something is weird with the first process that logs - its timer is
    #     # usually shorter than the rest, so we pick the max pid
    #     target_pid = max(run.target_pids[timer])
    #     target_pid_map[timer] = target_pid
    for timer, times in run.raw_timers.items():
        filtered_timers[timer] = []
        for pid, startend in zip(run.timer_pids[timer], times):
            if True or pid == target_pid_map[timer]:
                filtered_timers[timer].append(startend)
    print(f"Filtered timers to target PIDs: {target_pid_map}")
    
    # Filter out of range and deduplicate
    def filt_and_dedup(vals):
        return list(filter(lambda start_end: start_end[0] >= START and start_end[1] <= END if len(start_end) == 2 else start_end[0] >= START and start_end[0] <= END, vals))
    for timer in filtered_timers:
        print(f"Before filtering, timer '{timer}' has {len(run.raw_timers[timer])} entries")
        filtered_timers[timer] = filt_and_dedup(filtered_timers[timer])
        print(f"After filtering, timer '{timer}' has {len(filtered_timers[timer])} entries")
        for i in range(len(filtered_timers[timer])):
            for j in range(len(filtered_timers[timer][i])):
                filtered_timers[timer][i][j] = pd.to_datetime((filtered_timers[timer][i][j] - START).total_seconds(), unit='s')
    run.timers = filtered_timers
    starts = []
    ends = []
    durations = []
    kind = []
    for timer, times in run.timers.items():
        for start, end in times:
            starts.append(start)
            ends.append(end)
            durations.append((end - start).total_seconds())
            kind.append(timer)

    run.timers_df = pd.DataFrame({
        "start": starts,
        "end": ends,
        "duration": durations,
        "kind": kind
    })
    
    assert run.raw_data, f"Run {runid} is missing NSYS data, cannot plot GPU metrics"
    run.data = {gpu_id: {mname: series.between_time(START.time(), END.time()) for mname, series in mdict.items()} for gpu_id, mdict in run.raw_data.items()}
    # print(runid, run.data)
    for gpu_id, mdict in run.data.items():
        for mname, series in mdict.items():
            delta = series.index - START
            run.data[gpu_id][mname].index = pd.to_datetime(delta.total_seconds(), unit='s')
            # aucs[gpu_id][mname] = np.trapz(series.values, x=delta.total_seconds())
            # auc_by_timer = defaultdict(list)
            
    run.nsmi_metrics = run.raw_nsmi_metrics.between_time(START.time(), END.time())
    run.cpu_metrics = run.raw_cpu_metrics.between_time(START.time(), END.time())

Processing run sql_7b_4gpu_debugoom
defaultdict(<class 'list'>, {})
First timer at 2026-04-29 02:25:47.441694376, last timer at 2026-04-29 02:28:58.223328689
Base time at 2026-04-29 02:25:47.441694376
Visualizing data in this range: [2026-04-29 02:25:47.441694376, 2026-04-29 02:43:47.441694376]
As a seconds offset: [0.0, 1080.0]
Whole application took 0 days 00:03:10.781634313 seconds



Filtered timers to target PIDs: {}
Before filtering, timer 'generate' has 1 entries
After filtering, timer 'generate' has 1 entries
Before filtering, timer 'sync_weights' has 1 entries
After filtering, timer 'sync_weights' has 1 entries
Before filtering, timer 'fwd_logprobs_values_reward' has 1 entries
After filtering, timer 'fwd_logprobs_values_reward' has 1 entries
Before filtering, timer 'policy_train' has 1 entries
After filtering, timer 'policy_train' has 1 entries
Before filtering, timer 'train_fwd' has 8 entries
After filtering, timer 'train_fwd' has 8 entries
Before filtering, timer 'train_bwd'

In [15]:
# PLOT

# (metric_name, display_name)
METRICS_TO_PLOT = [
    ("Tensor Active [Throughput %]", "Tensor Core Utilization (%)"),
    # ("GR Active [Throughput %]", "GR Active [Throughput %]"),
    # ("SMs Active [Throughput %]", "SMs Active [Throughput %]"),
    # ("SM Issue [Throughput %]", "SM Issue [Throughput %]"),
    # ("Compute Warps in Flight [Throughput %]", "Compute Warps in Flight [Throughput %]"),
    # ("DRAM Read Bandwidth [Throughput %]", "DRAM Read Bandwidth [Throughput %]"),
    # ("DRAM Write Bandwidth [Throughput %]", "DRAM Write Bandwidth [Throughput %]"),
    # ("PCIe RX Throughput [Throughput %]", "PCIe RX Throughput [Throughput %]"),
    # ("PCIe TX Throughput [Throughput %]", "PCIe TX Throughput [Throughput %]"),
]

NSMI_METRICS_TO_PLOT = ["mem_alloc", "pcie_tx", "pcie_rx"]
NSMI_METRICS_TO_PLOT = ["mem_alloc"]

ROLLING_WINDOW = '1s'

swe_color_map = {
    "query": "tab:blue",
    "query_err": "tab:cyan",
    "get_observation": "tab:red",
    "get_observation_err": "tab:pink",
    "env_setup": "tab:green",
    "evaluation": "tab:orange"
}
swe_color_map = {
    "query": "lightblue",
    "query_err": "darkblue",
    "get_observation": "lightcoral",
    "get_observation_err": "darkred",
    "env_setup": "green",
    "evaluation": "orange"
}
timer_kind = "vllm"
for runid, run in RUNS.items():
    if timer_kind == "swe":
        # fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 8))
        swe_plot_data = {}
        unknown_tasks = set()
        known_tasks = set()
        for agent_name, timers in filter(lambda x: True, run.swe_finished_timers.items()):
            swe_plot_data[agent_name] = []
            for task_name, start, end in timers:
                if task_name not in swe_color_map:
                    unknown_tasks.add(task_name)
                    continue
                known_tasks.add(task_name)
                duration = (end - start).total_seconds()
                start = (start - run.first_timer).total_seconds()
                color = swe_color_map[task_name]
                # Store task_name so plotting function can use it for legend
                swe_plot_data[agent_name].append((task_name, color, start, duration))
        
        print(f"Unknown SWE tasks found: {unknown_tasks}")
        print(f"Known SWE tasks found: {known_tasks}")
        # plot_broken_bars(swe_plot_data, f"{runid} SWE Agent Timers", "Time", ax)
        fig = plot_broken_bars_plotly(swe_plot_data, f"{runid} SWE Agent Timers")
        fig.write_html("swe_timers.html")
        fig.show()

    # elif len(run.swe_finished_timers) > 0:
    #     print(list(run.swe_finished_timers.keys()))
    #     fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 12))
    #     durations = []
    #     for _, duration in run.swe_finished_timers["query"]:
    #         durations.append(duration.total_seconds())
    #     plot_histogram(ax[0], durations, f"{runid} SWE Query Duration", "Duration (seconds)", "Number of Occurences")
        
    #     durations = []
    #     for _, duration in run.swe_finished_timers["get_observation"]:
    #         durations.append(duration.total_seconds())
    #     plot_histogram(ax[1], durations, f"{runid} SWE Get Observation Duration", "Duration (seconds)", "Number of Occurences")
        
    #     durations = []
    #     for _, duration in run.swe_finished_timers["evaluation"]:
    #         durations.append(duration.total_seconds())
    #     plot_histogram(ax[2], durations, f"{runid} SWE Evaluation Duration", "Duration (seconds)", "Number of Occurences")
    #     plt.show()
    elif timer_kind == "trajectory":
        fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(12, 12))
        # run.vllm_finished_durations = []
        # for req_id, start_time, end_time in run.vllm_finished_timers:
        #     run.vllm_finished_durations.append((end_time - start_time).total_seconds())
        # print(run.vllm_finished_durations)
        # plot_histogram(ax[0, 0], run.vllm_finished_durations, f"{runid} VLLM Latency", "Duration (seconds)", "Count")
        trajectory_durations = []
        for traj_id, start_time, end_time in run.trajectory_timers:
            trajectory_durations.append((end_time - start_time).total_seconds())
        print(trajectory_durations)
        plot_histogram(ax[0, 1], trajectory_durations, f"{runid} Trajectory Latency", "Duration (seconds)", "Count")

        # Broken bars (Gantt) visualization: one lane per trajectory
        traj_plot_data = {}
        for traj_id, start_time, end_time in run.trajectory_timers:
            duration = (end_time - start_time).total_seconds()
            start_offset = (start_time - run.first_timer).total_seconds()
            traj_plot_data[traj_id] = [("trajectory", "steelblue", start_offset, duration)]
        fig_gantt = plot_broken_bars_plotly(traj_plot_data, f"{runid} Trajectory Timers")
        fig_gantt.write_html("trajectory_timers.html")
        fig_gantt.show()
    elif timer_kind == "vllm":
        print(run.vllm_finished_timers)
        # Group vllm requests by gpu_id: Dict[gpu_id, List[(task_name, color, start_sec, duration_sec)]]
        # Assign a distinct color per unique req_id using a colormap
        all_req_ids = sorted(set(req_id for _, req_id, _, _ in run.vllm_finished_timers), key=request_id_sort_key)
        _cmap = plt.get_cmap("tab20", 20)
        def _rgba(c):
            r, g, b, a = c
            return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{a:.2f})"
        def get_rid_index(req_id):
            # Extract the rollout number from the req_id for sorting
            try:
                return int(req_id.split("_")[0])
            except (ValueError, IndexError):
                return 0
        req_color_map = {rid: _rgba(_cmap(get_rid_index(rid) % _cmap.N)) for rid in all_req_ids}
        
        def retain_req_id(req_id):
            rollout = get_rid_index(req_id)
            return True
            return rollout > 65 and rollout < 85
        vllm_plot_data = defaultdict(list)
        for gpu_id, req_id, start_time, end_time in run.vllm_finished_timers:
            duration = (end_time - start_time).total_seconds()
            start_offset = (start_time - run.first_timer).total_seconds()
            if retain_req_id(req_id):
                vllm_plot_data[gpu_id].append((req_id, req_color_map[req_id], start_offset, duration))
        for req_id, start_time, end_time in run.train_forward_finished_timers:
            duration = (end_time - start_time).total_seconds()
            start_offset = (start_time - run.first_timer).total_seconds()
            # Add trajectory timers as black bars for reference
            for gpu_id in vllm_plot_data.keys():
                if retain_req_id(req_id):
                    vllm_plot_data[gpu_id].append((f"{req_id}", req_color_map[req_id], start_offset, duration))
        for req_id, start_time, end_time in run.train_forward_backward_finished_timers:
            duration = (end_time - start_time).total_seconds()
            start_offset = (start_time - run.first_timer).total_seconds()
            # Add trajectory timers as black bars for reference
            for gpu_id in vllm_plot_data.keys():
                if retain_req_id(req_id):
                    vllm_plot_data[gpu_id].append((f"{req_id}", req_color_map[get_rid_index(req_id)], start_offset, duration))
                
        extra_plots = defaultdict(list)
                
        gpu_ids = [(gpu_kind, gpu_id) for gpu_kind, gpu_id_list in run.gpus.items() for gpu_id in gpu_id_list]
        linestyles = ["solid", "dash", "dot", "dashdot"]
        alphas = [1.0, 0.75, 0.5, 0.25]
        # fig.update_yaxes(range=[0, 100], title_text="% utilization")
        # fig.update_xaxes(tickformat="%M:%S", title_text="Time (MM:SS)")
        def plotly_rgba(cmap_color, alpha=1.0):
            r, g, b, _ = cmap_color
            return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"
        for (gpu_kind, gpu_id), linestyle, alpha in zip(gpu_ids, linestyles, alphas):
            for i, (metric, display_name) in enumerate(METRICS_TO_PLOT):
                pos = i / len(METRICS_TO_PLOT)
                gpu_str = f"{gpu_kind.title()} GPU {gpu_id}"
                cmap_local = plt.get_cmap("tab10")
                # print(f"plotting metric {metric} for GPU {gpu_id} with linestyle {linestyle} and alpha {alpha}")
                series = run.data[gpu_id][metric]
                rolled = series.rolling(window=ROLLING_WINDOW).mean()
                rolled.index = (rolled.index - rolled.index[0]).total_seconds().astype(int)
                color = plotly_rgba(cmap_local(pos % cmap_local.N), alpha=alpha)
                print(f"Metric: {metric}")
                extra_plots[display_name].append(PlotData(metric_name=f"GPU{gpu_id}: {display_name}", color=color, line_style=linestyle, data=rolled))
            # for i, metric in enumerate(NSMI_METRICS_TO_PLOT):
            #     pos = i / len(NSMI_METRICS_TO_PLOT)
            #     gpu_str = f"{gpu_kind.title()} GPU {gpu_id}"
            #     cmap_local = plt.get_cmap("tab10")
            #     # print(f"plotting metric {metric} for GPU {gpu_id} with linestyle {linestyle} and alpha {alpha}")
            #     series = run.nsmi_metrics[run.nsmi_metrics["gpu"] == int(gpu_id)][metric]
            #     rolled = series.rolling(window=ROLLING_WINDOW).mean()
            #     rolled.index = (rolled.index - rolled.index[0]).total_seconds().astype(int)
            #     color = plotly_rgba(cmap_local(pos % cmap_local.N), alpha=alpha)
            #     extra_plots[metric].append(PlotData(metric_name=f"GPU{gpu_id}: {metric}", color=color, line_style=linestyle, data=rolled))
        vlines = []
        # for timer_type, readable_name in [("fwd_logprobs_values_reward", "forward"), ("generate", "generate"), ("sync_weights", "sync_weights"), ("train", "train")]:
        for timer_type, readable_name in [("fwd_logprobs_values_reward", "forward"), ("generate", "generate"), ("policy_train", "train")]:
            for start, end in run.timers[timer_type]:
                vlines.append((readable_name, start.timestamp(), end.timestamp()))
                
        #     pass
        # vlines.append(("inference", run.timers["fwd_logprobs_values_reward"][0][0].timestamp(), run.timers["fwd_logprobs_values_reward"][0][1].timestamp()))
        # vlines.append(("generate", run.timers["generate"][0][0].timestamp(), run.timers["generate"][0][1].timestamp()))
        # vlines.append(("sync_weights", run.timers["sync_weights"][0][0].timestamp(), run.timers["sync_weights"][0][1].timestamp()))
        # pprint(vlines)
        # if len(run.timers["policy_train"]) > 0:
        #     vlines.append(("policy_train", run.timers["policy_train"][0][0].timestamp(), run.timers["policy_train"][0][1].timestamp()))
        print(f"Adding {vlines} to the plot")
        if not vllm_plot_data and False:
            print(f"No vLLM timers found for run {runid}")
        else:
            print(f"Found vLLM timers for {len(vllm_plot_data)} GPU(s): {sorted(vllm_plot_data.keys())}")
            fig_vllm = plot_compact_gpu_rollouts(dict(vllm_plot_data), f"{runid} vLLM Request Timers", extra_plots=extra_plots, vlines=vlines)
            fig_vllm.write_html(f"{runid}_vllm_timers.html")
            fig_vllm.show()
            print(f"Plot saved to {runid}_vllm_timers.html")


[('None', 'generate-tokens-acb85a4191637d2e', Timestamp('2026-04-29 02:27:32.850000'), Timestamp('2026-04-29 02:27:37')), ('None', 'generate-tokens-ab290f0c200ea00a', Timestamp('2026-04-29 02:27:33.690000'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-a56292a2d53eaf5d', Timestamp('2026-04-29 02:27:33.420000'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-9634cead75f97cf9', Timestamp('2026-04-29 02:27:33'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-b014cfa34ccb0440', Timestamp('2026-04-29 02:27:32.980000'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-bd787c310267dc6e', Timestamp('2026-04-29 02:27:32.970000'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-9fa221b3c4492b03', Timestamp('2026-04-29 02:27:32.970000'), Timestamp('2026-04-29 02:27:38')), ('None', 'generate-tokens-9d52a0556a33e9cc', Timestamp('2026-04-29 02:27:33.560000'), Timestamp('2026-04-29 02:27:39')), ('None', 'generate-tokens-b3e45df5602b

KeyError: '6_0'

In [9]:
# PLOT NSYS DATA (Plotly)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# h for horizontally spread, v for vertically stacked, None for all separate figures
SUBFIGURE = "v"

cmap = plt.get_cmap("tab10")

for runid, run in RUNS.items():
    run.vline_config = {
        timer: (cmap(i), "-") for i, timer in enumerate(run.timers.keys())
    }

for runid, run in RUNS.items():
    assert run.raw_data, f"Run {runid} is missing NSYS data, cannot plot GPU metrics"
    if run.data:
        continue
    if run.timers:
        run.data = {gpu_id: {mname: series.between_time(START.time(), END.time()) for mname, series in mdict.items()} for gpu_id, mdict in run.raw_data.items()}
        # print(runid, run.data)
        for gpu_id, mdict in run.data.items():
            for mname, series in mdict.items():
                delta = series.index - START
                run.data[gpu_id][mname].index = pd.to_datetime(delta.total_seconds(), unit='s')
                # aucs[gpu_id][mname] = np.trapz(series.values, x=delta.total_seconds())
                # auc_by_timer = defaultdict(list)
    else:
        run.data = run.raw_data

cmap_plotly = plt.get_cmap("tab10")

def plotly_rgba(cmap_color, alpha=1.0):
    r, g, b, _ = cmap_color
    return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"

gpu_type_to_color_plotly = {
    "inference": "blue",
    "training": "red",
    "mixed": "purple",
}

def chart_gpu_metric_over_time_plotly(metric_group: list[str], chart_label: str):
    num_subplots = len(RUNS)

    if SUBFIGURE == "v":
        fig = make_subplots(
            rows=num_subplots, cols=1,
            shared_xaxes=True, shared_yaxes=True,
            subplot_titles=[run.run_kind for run in RUNS.values()],
            vertical_spacing=0.06,
        )
        def row_col(idx): return idx + 1, 1
    elif SUBFIGURE == "h":
        fig = make_subplots(
            rows=1, cols=num_subplots,
            shared_xaxes=True, shared_yaxes=True,
            subplot_titles=[run.run_kind for run in RUNS.values()],
            horizontal_spacing=0.06,
        )
        def row_col(idx): return 1, idx + 1
    else:
        raise ValueError(f"Unknown SUBFIGURE value: {SUBFIGURE}")

    end_times = []

    for fig_idx, (runid, run) in enumerate(RUNS.items()):
        row, col = row_col(fig_idx)
        gpu_ids = [(gpu_kind, gpu_id) for gpu_kind, gpu_id_list in run.gpus.items() for gpu_id in gpu_id_list]
        linestyles = ["solid", "dash", "dot", "dashdot"]
        alphas = [1.0, 0.75, 0.5, 0.25]

        end_time = None
        print(f"Plotting run {runid}")
        for (gpu_kind, gpu_id), linestyle, alpha in zip(gpu_ids, linestyles, alphas):
            gpu_str = f"{gpu_kind.title()} GPU {gpu_id}"
            cmap_local = plt.get_cmap("tab10")
            for i, metric in enumerate(metric_group):
                print(f"plotting metric {metric} for GPU {gpu_id} with linestyle {linestyle} and alpha {alpha}")
                series = run.data[gpu_id][metric]
                rolled = series.rolling(window=ROLLING_WINDOW).mean()
                # x_vals = rolled.index.astype(float)  # seconds since epoch (float)
                if end_time is None:
                    end_time = rolled.index.max()
                else:
                    end_time = max(end_time, rolled.index.max())

                fig.add_trace(
                    go.Scatter(
                        x=rolled.index,
                        y=rolled.values,
                        mode="lines",
                        name=f"{gpu_str} {metric}",
                        line=dict(color=plotly_rgba(cmap_local(i), alpha), dash=linestyle),
                        legendgroup=f"{gpu_str} {metric}",
                        showlegend=(fig_idx == 0),
                    ),
                    row=row, col=col,
                )

        end_times.append(end_time)

        # Shade timer regions / add vlines
        for t_idx, (timer, (color, _)) in enumerate(run.vline_config.items()):
            times = run.timers[timer]
            for k, start_end in enumerate(times):
                show_legend = (fig_idx == 0 and k == 0)
                if len(start_end) == 1:
                    fig.add_vline(
                        x=float(start_end[0].value) / 1e9,
                        line=dict(color=plotly_rgba(cmap_plotly(t_idx)), dash="solid", width=1),
                        opacity=0.3,
                        row=row, col=col,
                    )
                else:
                    start, end = start_end
                    x0, x1 = float(start.value) / 1e9, float(end.value) / 1e9
                    if timer != "policy_train":
                        fig.add_vrect(
                            x0=x0, x1=x1,
                            fillcolor=plotly_rgba(cmap_plotly(t_idx), 0.1),
                            line_width=0,
                            annotation_text=timer if show_legend else None,
                            annotation_position="top left",
                            row=row, col=col,
                        )

    # "Next step" green shading at the end
    max_end = max(e for e in end_times if e is not None)
    for fig_idx, (runid, run) in enumerate(RUNS.items()):
        row, col = row_col(fig_idx)
        e = end_times[fig_idx]
        if e is not None and e < max_end:
            fig.add_vrect(
                x0=e, x1=max_end,
                fillcolor="rgba(0,200,0,0.1)",
                line_width=0,
                annotation_text="Next step" if fig_idx == 0 else None,
                row=row, col=col,
            )

    fig.update_yaxes(range=[0, 100], title_text="% utilization")
    fig.update_xaxes(tickformat="%M:%S", title_text="Time (MM:SS)")
    fig.update_layout(
        height=500 * num_subplots if SUBFIGURE == "v" else 600,
        width=1200,
        title_text=chart_label,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()
    fig.write_html("nsys.html")

chart_gpu_metric_over_time_plotly(METRICS_TO_PLOT, "GPU Metrics Over Time")


Plotting run sql_7b_4gpu_debugoom
plotting metric ('Tensor Active [Throughput %]', 'Tensor Core Utilization (%)') for GPU 0 with linestyle solid and alpha 1.0


KeyError: ('Tensor Active [Throughput %]', 'Tensor Core Utilization (%)')

In [ ]:
# PLOT NSYS DATA
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.lines import Line2D
from matplotlib.dates import DateFormatter

gpu_type_to_color = {
    "inference": "blue",
    "training": "red",
    "mixed": "purple",
}
def chart_gpu_metric_over_time(metric_group: list[str], chart_label: str):
    in_legend = set()
    # min:sec:ms
    date_format = DateFormatter("%M:%S")

    if SUBFIGURE == "h":
        num_subplots = len(RUNS)
        fig, axes = plt.subplots(1, num_subplots, figsize=(12 * num_subplots, 12), sharex=True, sharey=True)
    elif SUBFIGURE == "v":
        num_subplots = len(RUNS)
        fig, axes = plt.subplots(num_subplots, 1, figsize=(12 * num_subplots, 12), sharex=True, sharey=True)
    else:
        assert SUBFIGURE is None
        num_subplots = 1
        assert False # Not implemented

    fig.supylabel(f"% utilization", fontsize=22)
    fig.supxlabel(f"Time (MM:SS)", fontsize=22)

        # num_subplots = 1
        # fig, axes = plt.subplots(num_subplots, 1, figsize=(4, 4 * num_subplots), sharex=True, sharey=True)
    # time_overlap_data = pd.concat([run.data[gpu_id][metric].rename(str(gpu_id)) for gpu_id in sorted(GPU_IDS.values())], axis=1).interpolate()
    # # print(time_overlap_data)
    # for gpu_type, gpu_ids in GPU_TYPES.items():
    #     time_overlap_data[f"{gpu_type}_on"] = False
    #     for gpu_id in gpu_ids:
    #         time_overlap_data[f"{gpu_type}_on"] |= time_overlap_data[str(gpu_id)] > 1
    # if len(TRAIN_GPUS) > 0 and len(INFERENCE_GPUS) > 0:
    #     time_overlap_data["train_and_infer"] = time_overlap_data["training_on"] & time_overlap_data["inference_on"]

    handles = [
         mpatches.Patch(color='green', alpha=0.1, label='Next step'),
        #  Line2D([0], [0], color="red", lw=1, label="GPU training"),
        #  Line2D([0], [0], color="blue", lw=1, label="GPU inference"),
    ]

    end_times = [None for _ in range(len(RUNS))]
    max_xlim = None
    for (fig_idx, run), tag in zip(enumerate(RUNS.values()), ["(a)", "(b)", "(c)"]):
        ax = axes[fig_idx] if num_subplots > 1 else axes
        # ax.set_xlim(pd.to_datetime("1970-01-01 00:00:00"), pd.to_datetime("1970-01-01 00:15:40"))
        ax.text(0, 101.5, tag, fontsize=22, verticalalignment='top')
        ax.set_title(f"{run.run_kind}", fontsize=22)
        gpu_ids = [(gpu_kind, gpu_id) for gpu_kind, gpu_id_list in run.gpus.items() for gpu_id in gpu_id_list]
        if fig_idx == 0:
            for (gpu_kind, gpu_id), linestyle, alpha in zip(gpu_ids, ["-", "--", ":", "-."], [1, 0.75, 0.5, 0.25]):
                handles.append(Line2D([0], [0], color="blue", lw=1, linestyle=linestyle, alpha=alpha, label=f"GPU {gpu_id} gen."))
            for (gpu_kind, gpu_id), linestyle, alpha in zip(gpu_ids, ["-", "--", ":", "-."], [1, 0.75, 0.5, 0.25]):
                handles.append(Line2D([0], [0], color="red", lw=1, linestyle=linestyle, alpha=alpha, label=f"GPU {gpu_id} tr."))

        end_time = None
        for (gpu_kind, gpu_id), linestyle, alpha in zip(gpu_ids, ["-", "--", ":", "-."], [1, 0.75, 0.5, 0.25]):
            gpu_str = f"{gpu_kind.title()} GPU {gpu_id}"
            cmap = plt.get_cmap("tab10")
            for i, metric in enumerate(metric_group):
                series = run.data[gpu_id][metric]
                if end_time is None:
                    end_time = series.index[-1]
                else:
                    end_time = max(end_time, series.index[-1])
                ax.plot(series.index, series.rolling(window=ROLLING_WINDOW).mean(), label=f"{gpu_str} {metric}", linestyle=linestyle, color=cmap(i), alpha=alpha)
            # ======  TODO: Keep the below code as it allows plotting multicolored lines
            # if gpu_kind != "mixed":
            #     ax.plot(series.index, series.rolling(window=ROLLING_WINDOW).mean(), label=gpu_str, linestyle=linestyle, color=gpu_type_to_color[gpu_kind], alpha=alpha)
            #     continue
            # for i, (gen_begin, gen_end) in enumerate(run.timers["generate"]):
            #     gpu_str = f"Mixed GPU {gpu_id}"
            #     generate_series = series[gen_begin:gen_end]
            #     print(f"Plotting generate series for {runid} {gpu_str} from {gen_begin} to {gen_end}, length {len(generate_series)}")
            #     ax.plot(generate_series.index, generate_series.rolling(window=ROLLING_WINDOW).mean(), label=gpu_str + f" during {gpu_kind}", linestyle=linestyle, color="blue", alpha=alpha)
            #     train_end = run.timers["generate"][i + 1][0] if i + 1 < len(run.timers["generate"]) else series.index[-1]
            #     train_series = series[gen_end:train_end]
            #     ax.plot(train_series.index, train_series.rolling(window=ROLLING_WINDOW).mean(), label=gpu_str + f" during {gpu_kind}", linestyle=linestyle, color="red", alpha=alpha)
            # for timer in run.timers:
            #     times = run.timers[timer]
            #     gpu_kind = timer_to_gpu_type.get(timer, "mixed")
            #     color = gpu_type_to_color[gpu_kind]
            #     for start, end in times:
            #         sliced_series = series[start:end]
            #         ax.plot(sliced_series.index, sliced_series.rolling(window=ROLLING_WINDOW).mean(), label=gpu_str + f" during {gpu_kind}", linestyle=linestyle, color=color, alpha=alpha)
        for timer, (color, linestyle) in run.vline_config.items():
            times = run.timers[timer]
            label = timer
            for start_end in times:
                if label in in_legend:
                    label = None
                else:
                    in_legend.add(label)
                if len(start_end) == 1:
                    time = start_end[0]
                    ax.axvline(time, color=color, linestyle=linestyle, alpha=0.3, label=label)
                    continue
                start, end = start_end
                if timer != "policy_train":
                    ax.fill_betweenx((0, 100), start, end, color=color, alpha=0.1, label=label)
        end_times[fig_idx] = end_time
        if max_xlim is None:
            max_xlim = ax.get_xlim()[1]
        else:
            max_xlim = max(max_xlim, ax.get_xlim()[1])
    for fig_idx, run in enumerate(RUNS.values()):
        ax = axes[fig_idx] if num_subplots > 1 else axes
        ax.fill_betweenx((0, 100), end_times[fig_idx], max_xlim, color="green", alpha=0.1)
        xlim_width = ax.get_xlim()[1] - ax.get_xlim()[0]
        padding = xlim_width * 0.02
        ax.set_xlim(0 - padding, max_xlim + padding)
        ax.tick_params(axis='both', which='major', labelsize=16)
        ax.tick_params(axis='both', which='minor', labelsize=16)
    plt.legend()
    # plt.legend(loc="upper right", handles=handles, fontsize=14)
    plt.gca().xaxis.set_major_formatter(date_format)
    # plt.gca().xaxis.set_minor_locator(AutoMinorLocator(5))
    plt.tight_layout()
    plt.savefig("plot.png", format="png")
    plt.savefig("plot.svg", format="svg")
    plt.show()

# def chart_cpu_usage_over_time():
#     date_format = DateFormatter("%M:%S")
#     in_legend = set()

#     fig, ax = plt.subplots(figsize=(12, 4))
#     # ax.set_xlim(pd.to_datetime("1970-01-01 00:00:00"), pd.to_datetime("1970-01-01 00:15:40"))
#     ax.title.set_text(f"CPU Usage Over Time")
#     ax.plot(windowed_cpu_sum.index, windowed_cpu_sum.rolling(window=ROLLING_WINDOW).mean(), label="Total CPU Usage", color="green")
#     for timer, (color, linestyle) in vline_config.items():
#         times = timers[timer]
#         label = timer
#         for start_end in times:
#             if label in in_legend:
#                 label = None
#             else:
#                 in_legend.add(label)
#             if len(start_end) == 1:
#                 time = start_end[0]
#                 ax.axvline(time, color=color, linestyle=linestyle, alpha=0.3, label=label)
#                 continue
#             start, end = start_end
#             ax.fill_betweenx((0, 5), start, end, color=color, alpha=0.1, label=label)
#     ax.legend(loc="upper right")

#     plt.gca().xaxis.set_major_formatter(date_format)
#     plt.gca().xaxis.set_minor_locator(AutoMinorLocator(10))
#     plt.tight_layout()
#     plt.show()

chart_gpu_metric_over_time(METRICS_TO_PLOT, "TODO")

In [ ]:
for runid, run in RUNS.items():
    print(f"====Run {runid} timers:====\n")
    
    print(run.timers_df[run.timers_df["kind"] == "optim_step"])
    # print(run.timers_df[run.timers_df["kind"] == "train_fwd"])
    
    duration_groups = run.timers_df.groupby("kind", sort=True)["duration"]
    
    print(duration_groups.count())
    # print()
    # print(duration_groups.sum())
    # print()
    print(duration_groups.mean())
    print(duration_groups.quantile(0.5))